In [4]:
import pandas as pd
import numpy as np

train=pd.read_csv("twitter-entity-sentiment-analysis/twitter_training.csv",header=None, names=['ID', 'Entity', 'Label','Text'])
val=pd.read_csv("twitter-entity-sentiment-analysis/twitter_validation.csv",header=None, names=['ID', 'Entity', 'Label','Text'])

train.head()

,ID,Entity,Label,Text
0,2401,Borderlands,Positive,im getting on borderlands and i will murder yo...
1,2401,Borderlands,Positive,I am coming to the borders and I will kill you...
2,2401,Borderlands,Positive,im getting on borderlands and i will kill you ...
3,2401,Borderlands,Positive,im coming on borderlands and i will murder you...
4,2401,Borderlands,Positive,im getting on borderlands 2 and i will murder ...


In [5]:
train.info()

<class 'pandas.DataFrame'>
RangeIndex: 74682 entries, 0 to 74681
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   ID      74682 non-null  int64
 1   Entity  74682 non-null  str  
 2   Label   74682 non-null  str  
 3   Text    73996 non-null  str  
dtypes: int64(1), str(3)
memory usage: 2.3 MB


In [6]:
train['Text'].isnull().sum()

np.int64(686)

In [7]:
train.isnull().sum()

ID          0
Entity      0
Label       0
Text      686
dtype: int64

In [8]:
train=train.dropna()
train.isnull().sum()

ID        0
Entity    0
Label     0
Text      0
dtype: int64

In [9]:
val.isnull().sum()

ID        0
Entity    0
Label     0
Text      0
dtype: int64

In [10]:
train["Label"].value_counts(normalize=True)

Label
Negative      0.302151
Positive      0.279137
Neutral       0.244716
Irrelevant    0.173996
Name: proportion, dtype: float64

In [11]:
val["Label"].value_counts(normalize=True)

Label
Neutral       0.285
Positive      0.277
Negative      0.266
Irrelevant    0.172
Name: proportion, dtype: float64

In [12]:
train[train["Label"]=="Irrelevant"].head()

,ID,Entity,Label,Text
102,2418,Borderlands,Irrelevant,Appreciate the (sonic) concepts / praxis Valen...
103,2418,Borderlands,Irrelevant,Appreciate the (sound) concepts / practices th...
104,2418,Borderlands,Irrelevant,Evaluate the (sound) concepts / concepts of Va...
105,2418,Borderlands,Irrelevant,Appreciate the (sonic) concepts / praxis Valen...
106,2418,Borderlands,Irrelevant,Appreciate by the ( sonic ) electronic concept...


In [13]:
import re
import string

def remove_urls(text):
    return re.sub(r'http\S+', '', text)

def remove_mentions(text):
    return re.sub(r'@\w+', '', text)

def remove_hashtags(text):
    return re.sub(r'#\w+', '', text)

def remove_punctuation(text):
    return text.translate(str.maketrans('', '', string.punctuation))

def lowercase(text):
    return text.lower()

def clean_text(text):
    text = remove_urls(text)
    text = remove_mentions(text)
    text = remove_hashtags(text)
    text = remove_punctuation(text)
    text = lowercase(text)
    return text


In [14]:
train['Text'] = train['Text'].apply(clean_text)
train.head()

,ID,Entity,Label,Text
0,2401,Borderlands,Positive,im getting on borderlands and i will murder yo...
1,2401,Borderlands,Positive,i am coming to the borders and i will kill you...
2,2401,Borderlands,Positive,im getting on borderlands and i will kill you all
3,2401,Borderlands,Positive,im coming on borderlands and i will murder you...
4,2401,Borderlands,Positive,im getting on borderlands 2 and i will murder ...


In [15]:
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\zarab\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [16]:
import re
import string

def remove_urls(text):
    return re.sub(r'http\S+', '', text)

def remove_mentions(text):
    return re.sub(r'@\w+', '', text)

def remove_hashtags(text):
    return re.sub(r'#\w+', '', text)

def remove_punctuation(text):
    return text.translate(str.maketrans('', '', string.punctuation))

def lowercase(text):
    return text.lower()

def clean_text(text):
    text = remove_urls(text)
    text = remove_mentions(text)
    text = remove_hashtags(text)
    text = remove_punctuation(text)
    text = lowercase(text)
    text=remove_stopword(text)
    return text


def remove_stopword(text):
    words=text.split()
    words = [word for word in words if word not in stop_words]
    return ' '.join(words)



In [17]:
train['Text'] = train['Text'].apply(clean_text)
train.head()


,ID,Entity,Label,Text
0,2401,Borderlands,Positive,im getting borderlands murder
1,2401,Borderlands,Positive,coming borders kill
2,2401,Borderlands,Positive,im getting borderlands kill
3,2401,Borderlands,Positive,im coming borderlands murder
4,2401,Borderlands,Positive,im getting borderlands 2 murder


In [18]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfid=TfidfVectorizer()
train_scored=tfid.fit_transform(train["Text"])
val_scored = tfid.transform(val["Text"])
y_train = train['Label']
y_val = val['Label']
print(train_scored.shape)

(73996, 39605)


In [20]:
from sklearn.linear_model import LogisticRegression


logreg=LogisticRegression(random_state=42,max_iter=1000)
logreg.fit(train_scored,y_train)

y_pred = logreg.predict(val_scored)

In [30]:
from sklearn.metrics import precision_score,f1_score,accuracy_score,recall_score
precision=precision_score(y_val,y_pred,average="weighted")
accuracy=accuracy_score(y_val,y_pred)
f1=f1_score(y_val,y_pred,average="weighted")
recall=recall_score(y_val,y_pred,average="weighted")

print(f"F1 score:{f1:4f}")
print(f"Accuracy score:", accuracy)
print(f"Precision score:{precision:.4f}")
print(f"REcall", recall)


F1 score:0.851363
Accuracy score: 0.852
Precision score:0.8557
REcall 0.852


In [32]:
from sklearn.naive_bayes import MultinomialNB

naive=MultinomialNB()
naive.fit(train_scored,y_train)
y_pred=naive.predict(val_scored)

In [33]:
from sklearn.metrics import precision_score,f1_score,accuracy_score,recall_score
precision=precision_score(y_val,y_pred,average="weighted")
accuracy=accuracy_score(y_val,y_pred)
f1=f1_score(y_val,y_pred,average="weighted")
recall=recall_score(y_val,y_pred,average="weighted")

print(f"F1 score:{f1:4f}")
print(f"Accuracy score:", accuracy)
print(f"Precision score:{precision:.4f}")
print(f"REcall", recall)


F1 score:0.771962
Accuracy score: 0.777
Precision score:0.7934
REcall 0.777


In [34]:
from sklearn.svm import LinearSVC

svm=LinearSVC(random_state=42)
svm.fit(train_scored,y_train)
y_pred=svm.predict(val_scored)

In [35]:
from sklearn.metrics import precision_score,f1_score,accuracy_score,recall_score
precision=precision_score(y_val,y_pred,average="weighted")
accuracy=accuracy_score(y_val,y_pred)
f1=f1_score(y_val,y_pred,average="weighted")
recall=recall_score(y_val,y_pred,average="weighted")

print(f"F1 score:{f1:4f}")
print(f"Accuracy score:", accuracy)
print(f"Precision score:{precision:.4f}")
print(f"REcall", recall)


F1 score:0.862426
Accuracy score: 0.863
Precision score:0.8692
REcall 0.863
